In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./sample30.csv')
print(df.head())
df['user_id_encoded'] = df['reviews_username'].astype('category').cat.codes
df['item_id_encoded'] = df['id'].astype('category').cat.codes
df['relation_encoded'] = 0

df['rating'] = df['reviews_rating']
df['reviews_text'] = df['reviews_text'].fillna("")




                     id            brand  \
0  AV13O1A8GV-KLJ3akUyj  Universal Music   
1  AV14LG0R-jtxr-f38QfS         Lundberg   
2  AV14LG0R-jtxr-f38QfS         Lundberg   
3  AV16khLE-jtxr-f38VFn              K-Y   
4  AV16khLE-jtxr-f38VFn              K-Y   

                                          categories  \
0  Movies, Music & Books,Music,R&b,Movies & TV,Mo...   
1  Food,Packaged Foods,Snacks,Crackers,Snacks, Co...   
2  Food,Packaged Foods,Snacks,Crackers,Snacks, Co...   
3  Personal Care,Medicine Cabinet,Lubricant/Sperm...   
4  Personal Care,Medicine Cabinet,Lubricant/Sperm...   

                         manufacturer  \
0  Universal Music Group / Cash Money   
1                            Lundberg   
2                            Lundberg   
3                                 K-Y   
4                                 K-Y   

                                         name              reviews_date  \
0   Pink Friday: Roman Reloaded Re-Up (w/dvd)  2012-11-30T06:21:45.000Z   
1

In [3]:
df.head()

,id,brand,categories,manufacturer,name,reviews_date,reviews_didPurchase,reviews_doRecommend,reviews_rating,reviews_text,reviews_title,reviews_userCity,reviews_userProvince,reviews_username,user_sentiment,user_id_encoded,item_id_encoded,relation_encoded,rating
0,AV13O1A8GV-KLJ3akUyj,Universal Music,"Movies, Music & Books,Music,R&b,Movies & TV,Mo...",Universal Music Group / Cash Money,Pink Friday: Roman Reloaded Re-Up (w/dvd),2012-11-30T06:21:45.000Z,NaN,NaN,5,i love this album. it's very good. more to the...,Just Awesome,Los Angeles,NaN,joshua,Positive,11440,0,0,5
1,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",Lundberg,Lundberg Organic Cinnamon Toast Rice Cakes,2017-07-09T00:00:00.000Z,True,NaN,5,Good flavor. This review was collected as part...,Good,NaN,NaN,dorothy w,Positive,6974,1,0,5
2,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",Lundberg,Lundberg Organic Cinnamon Toast Rice Cakes,2017-07-09T00:00:00.000Z,True,NaN,5,Good flavor.,Good,NaN,NaN,dorothy w,Positive,6974,1,0,5
3,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",K-Y,K-Y Love Sensuality Pleasure Gel,2016-01-06T00:00:00.000Z,False,False,1,I read through the reviews on here before look...,Disappointed,NaN,NaN,rebecca,Negative,19327,2,0,1
4,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",K-Y,K-Y Love Sensuality Pleasure Gel,2016-12-21T00:00:00.000Z,False,False,1,My husband bought this gel for us. The gel cau...,Irritation,NaN,NaN,walker557,Negative,24205,2,0,1


In [4]:
import tensorflow as tf
from transformers import TFBertModel, BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

I0000 00:00:1790059641.846008    3026 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790059641.882171    3026 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/deven/miniconda3/envs/anchit/lib/python3.12/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/deven/miniconda3/envs/anchit/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenizati

In [5]:
def build_joint_explainability_model(num_entities, num_relations, vocab_size, max_seq_len = 128, embed_dim = 64):
    input_ids = tf.keras.Input(shape=(max_seq_len,), dtype="int32", name = 'input_ids')
    attn_mask = tf.keras.Input(shape=(max_seq_len,), dtype="int32", name = 'attention_mask')


    bert = TFBertModel.from_pretrained('bert-base-uncased')
    bert.trainable = False
    bert_output = bert(input_ids = input_ids, attention_mask = attn_mask)
    bert_polled = bert_output.pooler_output

    head_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'head_idx')
    rel_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'rel_idx')
    tail_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'tail_idx')

    entity_emb_layer = tf.keras.layers.Embedding(input_dim = num_entities, output_dim = embed_dim, name = 'entity_emb')
    relation_emb_layer = tf.keras.layers.Embedding(input_dim = num_relations, output_dim = embed_dim, name = 'relation_emb')



    h_emb = tf.keras.layers.Flatten()(entity_emb_layer(head_idx))
    r_emb = tf.keras.layers.Flatten()(relation_emb_layer(rel_idx))
    t_emb = tf.keras.layers.Flatten()(entity_emb_layer(tail_idx))

    merged = tf.keras.layers.concatenate([h_emb, r_emb, t_emb])


    fusion_dense = tf.keras.layers.Dense(256, activation= 'relu', name = 'fusion_dense')(merged)
    fusion_state = tf.keras.layers.Dropout(0.2)(fusion_dense)


    rating_output = tf.keras.layers.Dense(1, activation= 'linear', name = 'rating_output')(fusion_state)




    decoder_inputs = tf.keras.layers.Input(shape = (None,), dtype="int32", name = 'decoder_inputs')

    lstm_initial_state = tf.keras.layers.Dense(512, activation= 'relu', name = 'lstm_initial_state')(fusion_state)

    decoder_embedding = tf.keras.layers.Embedding(input_dim = vocab_size, output_dim = 256, name = 'decoder_embedding')(decoder_inputs)


    decoder_lstm = tf.keras.layers.LSTM(512,return_sequences=True, activation= 'relu', name = 'justification_lstm')
    lstm_out = decoder_lstm(decoder_embedding, initial_state=[lstm_initial_state, lstm_initial_state])


    justification_output = tf.keras.layers.Dense(vocab_size, activation = 'softmax', name = 'justification_output' )(lstm_out)

    model = tf.keras.Model(
        inputs = [input_ids, attn_mask, head_idx, rel_idx, tail_idx, decoder_inputs],
        outputs = [rating_output, justification_output],
        name="Explainable_Hybrid_Recommender",
    )

    return model

In [6]:
vocab_size = tokenizer.vocab_size
max_user_id = df['user_id_encoded'].max()
max_item_id = df['item_id_encoded'].max()


num_entities = max(max_user_id, max_item_id + 1)
num_relations = df['relation_encoded'].max() + 1
model = build_joint_explainability_model(num_entities, num_relations, vocab_size, 128, 64)


model.summary()


model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate= 2e-5),
              loss = {
                  'rating_output' : 'mse',
                  'justification_output' : 'sparse_categorical_crossentropy',
              },
              loss_weights = {
                  'rating_output' : 1,
                  'justification_output' : 0.5
              },
              metrics = {'rating_output': 'mae'}
              )



I0000 00:00:1790059647.109201    3026 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5262 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification mode

Model: "Explainable_Hybrid_Recommender"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 head_idx (InputLayer)       [(None, 1)]                  0         []                            
                                                                                                  
 rel_idx (InputLayer)        [(None, 1)]                  0         []                            
                                                                                                  
 tail_idx (InputLayer)       [(None, 1)]                  0         []                            
                                                                                                  
 entity_emb (Embedding)      (None, 1, 64)                1594432   ['head_idx[0][0]',            
                                                                     

In [7]:
import numpy as np
from transformers import BertTokenizer


def create_tf_dataset(df, tokenizer, max_len = 128, batch_size = 16, is_training = True):
    print(f'Tokenizinf {len(df)}  Reviews')
    tokenized = tokenizer(
        df['reviews_text'].tolist(),
        truncation = True,
        padding = 'max_length',
        max_length = max_len,
        return_tensors = 'np',
    )




    input_ids = tokenized['input_ids']
    attn_mask = tokenized['attention_mask']


    decoder_inputs = input_ids[:, :-1]
    justification_targets = input_ids[:, 1:]

    inputs = {
        'input_ids': input_ids,
        'attention_mask': attn_mask,
        'decoder_inputs': decoder_inputs,
        "head_idx": df['user_id_encoded'].values.reshape(-1, 1),
        "rel_idx": df['relation_encoded'].values.reshape(-1, 1),
        "tail_idx": df['item_id_encoded'].values.reshape(-1, 1),

    }

    targets = {
        'rating_output':df['rating'].values.reshape(-1, 1),
        "justification_output": justification_targets
    }

    dataset = tf.data.Dataset.from_tensor_slices((inputs, targets))
    if is_training:
        dataset = dataset.shuffle(buffer_size = len(df), seed = 42)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
df_shuffled = df.sample(frac = 1, random_state = 42).reset_index(drop = True)
split_idx = int(len(df_shuffled) * 0.8)


train_idx = df_shuffled.iloc[:split_idx]
val_idx = df_shuffled.iloc[split_idx:]

train_dataset = create_tf_dataset(train_idx, tokenizer, max_len = 128, batch_size = 16, is_training = True)

val_dataset = create_tf_dataset(val_idx, tokenizer, max_len = 128, batch_size = 16, is_training = False)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor = 'val_loss', patience = 3, restore_best_weights=True)],
)








/home/deven/miniconda3/envs/anchit/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Tokenizinf 24000  Reviews
Tokenizinf 6000  Reviews
Epoch 1/15


I0000 00:00:1790059660.337070    3102 service.cc:153] XLA service 0x70aa7156ea20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790059660.337097    3102 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5050 Laptop GPU, Compute Capability 12.0a (Driver: 13.4.0; Runtime: 13.0.0; Toolkit: 13.0.0; DNN: 9.25.1)
I0000 00:00:1790059660.342513    3102 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1790059660.366901    3102 cuda_dnn.cc:461] Loaded cuDNN version 92501
I0000 00:00:1790059660.423198    3102 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1500/1500 [==============================] - 412s 272ms/step - loss: 30377805824.0000 - rating_output_loss: 15.6737 - justification_output_loss: 60755611648.0000 - rating_output_mae: 3.8229 - val_loss: 14.5639 - val_rating_output_loss: 12.3006 - val_justification_output_loss: 4.5266 - val_rating_output_mae: 3.3719
Epoch 2/15
1500/1500 [==============================] - 391s 261ms/step - loss: 13.1068 - rating_output_loss: 11.1004 - justification_output_loss: 4.0127 - rating_output_mae: 3.2046 - val_loss: 11.6662 - val_rating_output_loss: 9.8071 - val_justification_output_loss: 3.7181 - val_rating_output_mae: 3.0109
Epoch 3/15
 518/1500 [=========>....................] - ETA: 4:01 - loss: 11.4301 - rating_output_loss: 9.4330 - justification_output_loss: 3.9940 - rating_output_mae: 2.9536 

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction



def evaluate_nlg_performance(model, val_dataset, tokenizer):
    print('Evaluating NLG')

    bleu_scores = []
    smoother = SmoothingFunction().method1
    for batch_inputs, batch_targets in val_dataset:
        predictions = model(batch_inputs, training = False)
        justification_logits = predictions[1]
        pred_token_ids = tf.argmax(justification_logits, axis = -1).numpy()
        true_token_ids = batch_targets['justification_output'].numpy()

        for i in range(0, len(true_token_ids)):
            true_text = tokenizer.decode(true_token_ids[i], skip_special_tokens=True).split()
            pred_text = tokenizer.decode(pred_token_ids[i], skip_special_tokens=True).split()
            if len(true_text) > 0:
               score = sentence_bleu([true_text], pred_text, smoothing_function=smoother)
               bleu_scores.append(score)
    final_score = np.mean(bleu_scores)
    print('Final Score = ', final_score)
    print("\n--- Example Generation ---")
    print(f"Target: {' '.join(true_text)}")
    print(f"Output: {' '.join(pred_text)}")

    return final_score
val_bleu = evaluate_nlg_performance(model, val_dataset, tokenizer)
print(f'BLEU Score = {val_bleu}')


